# OpenDrift workflow template: one forward simulation

This notebook shows the full workflow I used for the particle-transport part of the thesis, but for **one** simulation only:
plastic released from the coastal clean-up locations, forced by currents + SSH + waves + wind (**CSWaWi**), with the
age-dependent **Terminal Velocity Chain (TVC)**. The other runs (C, CS, CSWa and the Baseline runs) use the same code with
different readers / terminal velocity, so they are not repeated here.

Steps:
1. Prepare one forcing file (gap fill, land re-mask, resample to 1 km)
2. Load and clean the clean-up records
3. Move the release points into the sea
4. Set up the readers and the OceanDrift model
5. Run the simulation one year at a time
6. Particle summary and coastal grounding
7. Trajectory metrics and residence in the 25 nm zone

The data files are not included in the repository (see the README for where they come from).

## 0. Imports and settings

In [ ]:
import gc
import random
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import rasterio
import rioxarray  # noqa: F401  (adds the .rio accessor)
import matplotlib.pyplot as plt
from pyproj import Geod
from scipy.ndimage import distance_transform_edt
from scipy.stats import wilcoxon
from shapely import contains_xy
from shapely.geometry import Point
from shapely.ops import nearest_points

from opendrift.models.oceandrift import OceanDrift
from opendrift.readers import reader_netCDF_CF_generic, reader_global_landmask

In [ ]:
# Paths (change these to where your data is)
DATA_DIR = Path("data")
OUT_DIR = Path("outputs/CSWaWi_TVC")
OUT_DIR.mkdir(parents=True, exist_ok=True)

CLEANUP_FILE = DATA_DIR / "cleanup_records.xlsx"      # Zibel + Nature Trust Malta records (not public)
COASTLINE_FILE = DATA_DIR / "coastline_boundary.shp"  # Malta coastline
BOUNDARY_25NM = DATA_DIR / "malta_25nm_buffer.shp"   # 25 nm Fisheries Management Zone
LAND_FILE = DATA_DIR / "land_polygons.shp"           # coastline used to re-mask the forcing

CURRENTS_FILE = DATA_DIR / "forcing/currents_2017_2025_1km.nc"
SSH_FILE = DATA_DIR / "forcing/ssh_2017_2025_1km.nc"
WAVES_PATTERN = "forcing/waves_{year}_1km.nc"          # one file per year
WIND_PATTERN = "forcing/wind_{year}_1km.nc"            # one file per year
BATHYMETRY_FILE = DATA_DIR / "bathymetry_emodnet.tif"

RUN_NAME = "Plastic_TVC_CSWaWi_2017_2025"

# Records
LITTER_CATEGORY = "Plastic"
START_YEAR, END_YEAR = 2017, 2025
BUFFER_M = 150                 # release point is picked at random within 150 m of the record (in the sea)

# Model settings (Table 3.3)
TIME_STEP = 3600               # 1 h
OUTPUT_STEP = 6 * 3600         # save every 6 h
WINDAGE = 0.02                 # only used in this configuration because wind is switched on (CSWaWi)
CURRENT_DRIFT_FACTOR = 1.0
VERTICAL_DIFFUSIVITY = 0.001   # m2/s, constant
MIXING_STEP = 300              # s
SEED_DEPTH = 0.0

# Terminal Velocity Chain: each particle changes velocity with its own age (Table 3.6)
TV_PHASE_DAYS = np.array([0, 60, 120, 180, 240, 300], dtype=float)
TV_PHASE_VALUES = np.array([0.021, 0.0, -0.001, -0.005, -0.010, -0.020], dtype=np.float32)
# for the Baseline run use a single value instead: TV_PHASE_VALUES = np.array([0.021]*6)

GROUNDING_DEPTH_M = 1.0        # deactivations shallower than this are coastal grounding, not seabed

CRS_WGS84, CRS_METRIC = "EPSG:4326", "EPSG:32633"
random.seed(42)

## 1. Prepare a forcing file

The CMEMS fields have gaps along the coast, and OpenDrift deactivates particles that enter cells with no current data.
So the gaps are filled with the nearest valid cell (only in a box around Malta), land is masked again using a coastline
shrunk inwards by 4 km, and the field is then resampled to the 1/120° (~1 km) grid with bilinear interpolation.
This is shown for the currents; SSH, waves and wind were done the same way.

In [ ]:
FILL_BOX = dict(lon_min=12, lon_max=17, lat_min=34, lat_max=38)   # only fill gaps around Malta
LAND_BUFFER_M = 4000                                               # shrink land by 4 km before re-masking
TARGET_RES = 1 / 120                                               # ~1 km grid


def nearest_neighbour_fill(arr):
    """Fill NaNs with the value of the closest valid cell."""
    nan_mask = np.isnan(arr)
    if not nan_mask.any() or nan_mask.all():
        return arr
    _, idx = distance_transform_edt(nan_mask, return_distances=True, return_indices=True)
    out = arr.copy()
    out[nan_mask] = arr[idx[0][nan_mask], idx[1][nan_mask]]
    return out


def land_mask_on_grid(lat, lon, land_gdf):
    """True where the (shrunk) land polygons cover a grid cell."""
    dummy = xr.DataArray(np.ones((len(lat), len(lon))), coords={"y": lat, "x": lon}, dims=["y", "x"])
    dummy = dummy.rio.write_crs("EPSG:4326")
    clipped = dummy.rio.clip(land_gdf.geometry, land_gdf.crs, invert=True, drop=False)
    return np.isnan(clipped.values)


def prepare_field(da, land_gdf):
    """Fill -> re-mask -> resample for one 2D slice (lat x lon)."""
    lat, lon = da["latitude"].values, da["longitude"].values
    raw = da.values.astype("float32")
    filled = nearest_neighbour_fill(raw)

    # keep the original NaNs outside the fill box
    in_box = ((lon[None, :] >= FILL_BOX["lon_min"]) & (lon[None, :] <= FILL_BOX["lon_max"]) &
              (lat[:, None] >= FILL_BOX["lat_min"]) & (lat[:, None] <= FILL_BOX["lat_max"]))
    filled = np.where(in_box | ~np.isnan(raw), filled, np.nan)

    filled[land_mask_on_grid(lat, lon, land_gdf)] = np.nan

    out_lat = np.arange(lat.min(), lat.max(), TARGET_RES)
    out_lon = np.arange(lon.min(), lon.max(), TARGET_RES)
    da_f = xr.DataArray(filled, coords={"latitude": lat, "longitude": lon}, dims=["latitude", "longitude"])
    return da_f.interp(latitude=out_lat, longitude=out_lon, method="linear")  # bilinear, no extrapolation

In [ ]:
# Shrink the land polygons by 4 km so coastal cells keep their filled values
land = gpd.read_file(LAND_FILE).to_crs("EPSG:3035")
land["geometry"] = land.buffer(-LAND_BUFFER_M)
land = land[~land.geometry.is_empty].to_crs("EPSG:4326")

# Quick check on the first time step of the surface currents
raw_currents = xr.open_dataset(DATA_DIR / "forcing/cmems_currents_native.nc")
u0 = raw_currents["uo"].isel(time=0, depth=0)
u0_1km = prepare_field(u0, land)
print("NaNs before:", int(np.isnan(u0.values).sum()), " after (on 1 km grid):", int(np.isnan(u0_1km.values).sum()))
u0_1km.plot(cmap="RdBu_r", figsize=(9, 5))
plt.title("uo, filled and resampled (t = 0)")
plt.show()

# The full file was made by looping prepare_field() over every time step / depth / variable
# and writing the result to NetCDF with the CF attributes OpenDrift needs (standard_name, units)

## 2. Load the clean-up records

In [ ]:
df = pd.read_excel(CLEANUP_FILE)
df["Year"] = pd.to_numeric(df["Year"], errors="coerce")

# Plastic only, 2017-2025 (to match the current data)
df = df[(df["Category"] == LITTER_CATEGORY) & df["Year"].between(START_YEAR, END_YEAR)].copy()

# Months are written in different ways in the sheet (3, "March", "Mar")
MONTHS = {m.lower(): i for i, m in enumerate(
    ["January", "February", "March", "April", "May", "June", "July",
     "August", "September", "October", "November", "December"], start=1)}
MONTHS.update({k[:3]: v for k, v in MONTHS.items()})

def to_month(v):
    try:
        v = int(float(v))
        return v if 1 <= v <= 12 else np.nan
    except (TypeError, ValueError):
        return MONTHS.get(str(v).strip().lower(), np.nan)

df["Month_num"] = df["Month"].apply(to_month)
df = df.dropna(subset=["Latitude", "Longitude", "Year", "Month_num"]).copy()

# Every record is released on the 1st of its month
df["release_time"] = pd.to_datetime(dict(year=df["Year"].astype(int), month=df["Month_num"].astype(int), day=1))
df["source_id"] = np.arange(len(df))

print("Plastic records:", len(df))
print("Releases from", df.release_time.min().date(), "to", df.release_time.max().date())

In [ ]:
# Compass region of each record (bearing from the centre of the islands), used later to compare regions
CENTRE_LON, CENTRE_LAT = 14.400, 35.923
REGIONS = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]
geod = Geod(ellps="WGS84")

az, _, _ = geod.inv(np.full(len(df), CENTRE_LON), np.full(len(df), CENTRE_LAT),
                    df["Longitude"].values, df["Latitude"].values)
bearing = (az + 360) % 360
df["region"] = [REGIONS[int(((b + 22.5) % 360) // 45)] for b in bearing]
df["region"].value_counts().reindex(REGIONS, fill_value=0)

## 3. Move the release points into the sea

Most records are on the beach, and OpenDrift deactivates particles released on land. So:
(1) records on land are moved to the nearest point on the coastline, (2) a random point is picked in the sea within 150 m,
and (3) any point that OpenDrift's own (GSHHG) landmask still thinks is land is moved to the nearest water cell.

In [ ]:
coast = gpd.read_file(COASTLINE_FILE).to_crs(CRS_METRIC)
zone = gpd.read_file(BOUNDARY_25NM).to_crs(CRS_METRIC)
land_m = coast.geometry.union_all()
sea_m = zone.geometry.union_all()

pts = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.Longitude, df.Latitude), crs=CRS_WGS84).to_crs(CRS_METRIC)

# Step 1: Move records on land to the nearest coastline point
on_land = pts.within(land_m)
coast_line = land_m.boundary
pts.loc[on_land, "geometry"] = [nearest_points(p, coast_line)[1] for p in pts.loc[on_land, "geometry"]]
print("Records snapped to the coast:", int(on_land.sum()), "of", len(pts))


# Step 2: Select a random point at sea within 150 m
def random_point_in(poly, tries=5000):
    minx, miny, maxx, maxy = poly.bounds
    for _ in range(tries):
        p = Point(random.uniform(minx, maxx), random.uniform(miny, maxy))
        if poly.contains(p):
            return p
    return poly.representative_point()

sea_buffers = [g.buffer(BUFFER_M).intersection(sea_m).difference(land_m) for g in pts.geometry]
pts["geometry"] = [None if b.is_empty else random_point_in(b) for b in sea_buffers]
pts = pts.dropna(subset=["geometry"]).set_geometry("geometry").to_crs(CRS_WGS84)

seed_lon = pts.geometry.x.to_numpy()
seed_lat = pts.geometry.y.to_numpy()

In [ ]:
# Step 3: Check OpenDrift's land mask and move any points still on land
landmask = reader_global_landmask.Reader()

def is_land(lon, lat):
    return landmask.get_variables(["land_binary_mask"], x=np.atleast_1d(lon), y=np.atleast_1d(lat))["land_binary_mask"]

def nearest_water(lon, lat, radius=0.005, n=101):
    gx, gy = np.meshgrid(lon + np.linspace(-radius, radius, n), lat + np.linspace(-radius, radius, n))
    gx, gy = gx.ravel(), gy.ravel()
    water = ~is_land(gx, gy).astype(bool)
    if not water.any():
        return lon, lat, np.nan
    d = np.hypot(gx - lon, gy - lat)
    d[~water] = np.inf
    k = np.argmin(d)
    return gx[k], gy[k], d[k] * 111_000  # rough metres

flagged = np.where(is_land(seed_lon, seed_lat))[0]
moved = []
for i in flagged:
    seed_lon[i], seed_lat[i], dist = nearest_water(seed_lon[i], seed_lat[i])
    moved.append(dist)
print(f"Moved to OpenDrift water: {len(flagged)} points, median {np.nanmedian(moved):.0f} m, max {np.nanmax(moved):.0f} m")
print("Still on land:", int(is_land(seed_lon, seed_lat).sum()))

seed_time = pd.to_datetime(pts["release_time"]).to_numpy()
seed_id = pts["source_id"].to_numpy()
seed_region = pts["region"].to_numpy()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
coast.to_crs(CRS_WGS84).plot(ax=ax, color="lightgrey", edgecolor="black")
ax.scatter(seed_lon, seed_lat, s=10, c="tab:blue")
ax.set_title(f"{len(seed_lon)} release points")
plt.show()

## 4. Readers and model

Currents, SSH and bathymetry are single files. Waves and wind are read one year at a time (one file per year), which was
much faster than letting OpenDrift open all the files together.
OceanDrift is extended so the terminal velocity depends on each particle's age.

In [ ]:
reader_currents = reader_netCDF_CF_generic.Reader(str(CURRENTS_FILE))
reader_ssh = reader_netCDF_CF_generic.Reader(str(SSH_FILE))

def yearly_reader(pattern, year):
    return reader_netCDF_CF_generic.Reader(str(DATA_DIR / pattern.format(year=year)))

# bathymetry: crop the EMODnet raster to the currents domain and turn it into a reader
with rasterio.open(BATHYMETRY_FILE) as src:
    win = rasterio.windows.from_bounds(reader_currents.xmin - 0.25, reader_currents.ymin - 0.25,
                                       reader_currents.xmax + 0.25, reader_currents.ymax + 0.25,
                                       transform=src.transform)
    elev = src.read(1, window=win).astype(float)
    tr = src.window_transform(win)
lon_b = tr.c + (np.arange(elev.shape[1]) + 0.5) * tr.a
lat_b = tr.f + (np.arange(elev.shape[0]) + 0.5) * tr.e
depth = np.where(elev < 0, -elev, np.nan)          # positive depth, land = NaN
if lat_b[0] > lat_b[-1]:
    lat_b, depth = lat_b[::-1], depth[::-1]

ds_bathy = xr.Dataset({"sea_floor_depth_below_sea_level": (("latitude", "longitude"), depth)},
                      coords={"longitude": lon_b, "latitude": lat_b})
ds_bathy["sea_floor_depth_below_sea_level"].attrs = {"standard_name": "sea_floor_depth_below_sea_level", "units": "m"}
ds_bathy["longitude"].attrs = {"standard_name": "longitude", "units": "degrees_east"}
ds_bathy["latitude"].attrs = {"standard_name": "latitude", "units": "degrees_north"}
reader_bathy = reader_netCDF_CF_generic.Reader(ds_bathy, proj4="+proj=latlong +datum=WGS84")

print("Currents:", reader_currents.start_time, "to", reader_currents.end_time)

In [ ]:
def tv_for_age(age_seconds):
    """Terminal velocity for each particle from its age (step function, Table 3.6)."""
    phase = np.searchsorted(TV_PHASE_DAYS, age_seconds / 86400.0, side="right") - 1
    return TV_PHASE_VALUES[np.clip(phase, 0, len(TV_PHASE_VALUES) - 1)]


class LaggedOceanDrift(OceanDrift):
    """OceanDrift where every particle follows the TV chain from its own release time."""
    def update_terminal_velocity(self, Tprofiles=None, Sprofiles=None, z_index=None):
        self.elements.terminal_velocity[:] = tv_for_age(self.elements.age_seconds)


def make_model(waves, wind):
    o = LaggedOceanDrift(loglevel=20)
    o.add_reader([reader_currents, reader_ssh, waves, wind, reader_bathy])
    o.set_config("general:coastline_action", "previous")     # back to last position if it hits land
    o.set_config("general:seafloor_action", "deactivate")    # stop at the seabed
    o.set_config("seed:ocean_only", True)
    o.set_config("drift:stokes_drift", True)
    o.set_config("drift:vertical_advection", False)
    o.set_config("drift:vertical_mixing", True)
    o.set_config("drift:vertical_mixing_at_surface", True)
    o.set_config("vertical_mixing:diffusivitymodel", "constant")
    o.set_config("environment:fallback:ocean_vertical_diffusivity", VERTICAL_DIFFUSIVITY)
    o.set_config("vertical_mixing:timestep", MIXING_STEP)
    return o

## 5. Run one year at a time

Running all nine years in one go was very slow, so each calendar year is a separate run. Particles that are still active
at the end of a year are seeded again at their last position at the start of the next year, and their **age is put back**
so they stay on the right step of the TV chain. Chunk ends are kept on the 6 h output grid so the years join up.

In [ ]:
CHUNK_DIR = OUT_DIR / "_year_chunks"
CHUNK_DIR.mkdir(exist_ok=True)

n = len(seed_lon)
release = pd.to_datetime(seed_time)
sim_start = release.min()
sim_end = pd.Timestamp(reader_currents.end_time)

def on_grid(t):
    """Round down to the 6 h output grid that starts at sim_start."""
    k = int((t - sim_start).total_seconds() // OUTPUT_STEP)
    return sim_start + pd.Timedelta(seconds=k * OUTPUT_STEP)

# carried state between years
last_lon = np.full(n, np.nan); last_lat = np.full(n, np.nan); last_z = np.full(n, np.nan)
last_age = np.full(n, np.nan)
active = np.zeros(n, bool)
released = np.zeros(n, bool)
chunks = []

t0 = sim_start
for year in range(sim_start.year, sim_end.year + 1):
    t1 = on_grid(min(sim_end, pd.Timestamp(f"{year}-12-31 23:59")))
    if t1 <= t0:
        continue

    new = np.where(~released & (release >= t0) & (release <= t1))[0]
    carry = np.where(active)[0]
    idx = np.concatenate([carry, new]).astype(int)
    print(f"{year}: carried {len(carry)}, new {len(new)}")
    if len(idx) == 0:
        t0 = t1
        continue

    o = make_model(yearly_reader(WAVES_PATTERN, year), yearly_reader(WIND_PATTERN, year))
    o.seed_elements(
        lon=np.concatenate([last_lon[carry], seed_lon[new]]),
        lat=np.concatenate([last_lat[carry], seed_lat[new]]),
        z=SEED_DEPTH,   # all particles (new and carried) start each year's run at the surface
        time=list(pd.to_datetime(np.concatenate([np.full(len(carry), t0), release.values[new]])).to_pydatetime()),
        current_drift_factor=CURRENT_DRIFT_FACTOR,
        wind_drift_factor=WINDAGE,
        terminal_velocity=TV_PHASE_VALUES[0],
    )

    # seed_elements() resets age to 0, so put the real age back for carried particles
    if len(carry):
        sched = o.elements_scheduled
        if not isinstance(sched.age_seconds, np.ndarray):
            sched.age_seconds = np.full(len(idx), float(sched.age_seconds), dtype=np.float32)
        if not isinstance(sched.terminal_velocity, np.ndarray):
            sched.terminal_velocity = np.full(len(idx), float(sched.terminal_velocity), dtype=np.float32)
        sched.age_seconds[:len(carry)] = last_age[carry]
        sched.terminal_velocity[:len(carry)] = tv_for_age(last_age[carry])

    chunk_file = CHUNK_DIR / f"{RUN_NAME}_{year}.nc"
    o.run(end_time=t1.to_pydatetime(), time_step=TIME_STEP, time_step_output=OUTPUT_STEP,
          outfile=str(chunk_file), export_buffer_length=10,
          export_variables=["lon", "lat", "z", "status", "terminal_velocity", "age_seconds"])

    # save each particle's last state for the next year (status 0 = still active)
    with xr.open_dataset(chunk_file) as ds:
        lon, lat, z = ds["lon"].values, ds["lat"].values, ds["z"].values
        age, status = ds["age_seconds"].values, ds["status"].values
    for j, g in enumerate(idx):
        ok = np.where(np.isfinite(lon[j]))[0]
        if len(ok) == 0:
            active[g] = False
            continue
        k = ok[-1]
        last_lon[g], last_lat[g], last_z[g], last_age[g] = lon[j, k], lat[j, k], z[j, k], age[j, k]
        active[g] = status[j, k] == 0

    released[idx] = True
    chunks.append((chunk_file, idx))
    t0 = t1
    del o
    gc.collect()

print("Done. Still active at the end:", int(active.sum()))

In [ ]:
# stitch the yearly files into one particle x time array on the 6 h grid
times = pd.date_range(sim_start, t0, freq=f"{OUTPUT_STEP}s")
LON = np.full((n, len(times)), np.nan, np.float32); LAT = LON.copy(); Z = LON.copy()
STATUS = np.full((n, len(times)), -1, np.int16)

for f, idx in chunks:
    with xr.open_dataset(f) as ds:
        k = ((pd.to_datetime(ds["time"].values) - sim_start).total_seconds() // OUTPUT_STEP).astype(int)
        lon, lat, z, st = ds["lon"].values, ds["lat"].values, ds["z"].values, ds["status"].values
        flags = ds["status"].attrs.get("flag_meanings", "").split()
    for j, g in enumerate(idx):
        ok = np.isfinite(lon[j])
        LON[g, k[ok]], LAT[g, k[ok]], Z[g, k[ok]], STATUS[g, k[ok]] = lon[j, ok], lat[j, ok], z[j, ok], st[j, ok]

xr.Dataset({v: (("trajectory", "time"), a) for v, a in [("lon", LON), ("lat", LAT), ("z", Z), ("status", STATUS)]},
           coords={"time": times}).to_netcdf(OUT_DIR / f"{RUN_NAME}.nc")
print("Combined file saved:", LON.shape)

## 6. Particle summary and coastal grounding

The final position of each particle is its seabed location. Floating particles can also be stopped in very shallow coastal
cells (shallower than the SSH offset), so anything deactivated in less than 1 m of water is counted as **coastal grounding**
and left out of the seabed analyses.

In [ ]:
last = np.array([np.where(np.isfinite(LON[p]))[0][-1] if np.isfinite(LON[p]).any() else -1 for p in range(n)])
p_idx = np.arange(n)
final_status = np.array([flags[STATUS[p, k]] if k >= 0 else "no_output" for p, k in zip(p_idx, last)])

summary = pd.DataFrame({
    "source_id": seed_id, "region": seed_region, "release_time": release,
    "release_lon": seed_lon, "release_lat": seed_lat,
    "final_time": times[np.maximum(last, 0)],
    "final_lon": LON[p_idx, last], "final_lat": LAT[p_idx, last], "final_depth_m": Z[p_idx, last],
    "final_status": final_status,
})
summary["age_at_final_days"] = (summary.final_time - summary.release_time).dt.total_seconds() / 86400

seafloor = summary.final_status == "seafloor"
summary["outcome"] = np.where(~seafloor, summary.final_status,
                              np.where(summary.final_depth_m > -GROUNDING_DEPTH_M, "coastal_grounding", "seabed"))
summary.to_csv(OUT_DIR / f"{RUN_NAME}_Particle_Summary.csv", index=False)
summary["outcome"].value_counts()

## 7. Trajectory metrics and the 25 nm zone

Seven metrics for every particle from its 6-hourly positions (geodesic distances on WGS84), then residence time,
exits and re-entries for the 25 nm zone.

In [ ]:
def track_metrics(lon, lat, t):
    ok = np.isfinite(lon) & np.isfinite(lat)
    lon, lat, t = lon[ok], lat[ok], t[ok]
    if len(lon) < 3:
        return dict.fromkeys(["path_km", "net_km", "straightness", "duration_d", "speed_ms",
                              "turn_total_deg", "turn_mean_deg"], np.nan)
    az, _, d = geod.inv(lon[:-1], lat[:-1], lon[1:], lat[1:])
    path = d.sum()
    _, _, net = geod.inv(lon[0], lat[0], lon[-1], lat[-1])
    duration = (t[-1] - t[0]) / np.timedelta64(1, "s")
    turns = np.abs((np.diff(az[d > 0]) + 180) % 360 - 180)   # change in heading, -180..180
    return {"path_km": path / 1000, "net_km": net / 1000, "straightness": net / path if path > 0 else np.nan,
            "duration_d": duration / 86400, "speed_ms": path / duration if duration > 0 else np.nan,
            "turn_total_deg": turns.sum(), "turn_mean_deg": turns.mean() if len(turns) else np.nan}

metrics = pd.DataFrame([track_metrics(LON[p], LAT[p], times.values) for p in range(n)])
metrics.insert(0, "source_id", seed_id)
metrics["region"] = seed_region
metrics.to_csv(OUT_DIR / f"{RUN_NAME}_trajectory_metrics.csv", index=False)
metrics.describe().T[["50%", "25%", "75%"]]

In [ ]:
zone_poly = gpd.read_file(BOUNDARY_25NM).to_crs(CRS_WGS84).geometry.union_all()
step_days = OUTPUT_STEP / 86400

rows = []
for p in range(n):
    ok = np.isfinite(LON[p])
    if not ok.any():
        continue
    inside = contains_xy(zone_poly, LON[p, ok], LAT[p, ok])
    change = np.diff(inside.astype(int))
    exits, entries = np.where(change == -1)[0], np.where(change == 1)[0]
    rows.append({
        "source_id": seed_id[p],
        "residence_days": inside.sum() * step_days,
        "exited": len(exits) > 0,
        "reentered": len(exits) > 0 and (entries > exits.min()).any(),
        "permanent_exit": len(exits) > 0 and not inside[-1],
        "ended_inside": bool(inside[-1]),
    })
residence = pd.DataFrame(rows)
print(f"Median residence: {residence.residence_days.median():.1f} days")
print(f"Permanent exit: {100 * residence.permanent_exit.mean():.0f}%")
print(f"Re-entered (of those that left): {100 * residence.loc[residence.exited, 'reentered'].mean():.0f}%")

### Comparing with another configuration

The configurations were compared particle by particle (matched by `source_id`) with a Wilcoxon signed-rank test,
rank-biserial effect size and Holm correction across the three comparisons with C. This function is what I used,
once the metrics file of the other run is available.

In [ ]:
def holm(pvals):
    p = np.asarray(pvals, float)
    order = np.argsort(p)
    adj = np.empty_like(p)
    running = 0
    for rank, i in enumerate(order):
        running = max(running, (len(p) - rank) * p[i])
        adj[i] = min(running, 1.0)
    return adj


def compare_runs(ref, other, metric):
    """Wilcoxon signed-rank on matched particles + matched-pairs rank-biserial correlation."""
    m = ref[["source_id", metric]].merge(other[["source_id", metric]], on="source_id", suffixes=("_ref", "_other")).dropna()
    diff = m[f"{metric}_other"] - m[f"{metric}_ref"]
    diff = diff[diff != 0]
    stat, p = wilcoxon(diff)
    ranks = pd.Series(np.abs(diff)).rank()
    r_rb = (ranks[diff.values > 0].sum() - ranks[diff.values < 0].sum()) / ranks.sum()
    pct = (100 * (m[f"{metric}_other"] - m[f"{metric}_ref"]) / m[f"{metric}_ref"]).median()
    return {"metric": metric, "n": len(m), "W": stat, "p": p, "rank_biserial": r_rb, "median_change_pct": pct}

# example (needs the C run's metrics file):
# ref = pd.read_csv("outputs/C_TVC/Plastic_TVC_C_2017_2025_trajectory_metrics.csv")
# res = pd.DataFrame([compare_runs(ref, metrics, "path_km")])

## Quick map of the trajectories

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
for p in range(0, n, 2):           # every second particle so the map is readable
    ax.plot(LON[p], LAT[p], lw=0.4, alpha=0.4)
sb = summary[summary.outcome == "seabed"]
ax.scatter(sb.final_lon, sb.final_lat, s=8, c="black", label="seabed delivery", zorder=3)
gpd.read_file(BOUNDARY_25NM).to_crs(CRS_WGS84).boundary.plot(ax=ax, color="red", ls="--")
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
ax.set_title("CSWaWi, Terminal Velocity Chain")
ax.legend()
plt.savefig(OUT_DIR / f"{RUN_NAME}_trajectories.png", dpi=300, bbox_inches="tight")
plt.show()